In [ ]:
import os
import glob 
import arcpy
import math
import pandas as pd
from arcpy.sa import *
import arcpy
import demreconditioning
import fillsinks
import flowdirection_ah

def get_smoothened_shp_ind(smoothened_fc = r'.\final.gdb\catchment_and_snappoints\Smoothened',
                           save_dir = r'.\python_outputs\lfp_outputs\1_4_smoothened_catchment'):
    '''
    i. Using 'Shape@' in UpdateCursor in the function lfp_python and using it for clipping did not work, so this approach was 
    implemented.
    
    ii. This saves all individual shapefiles in a feature class in one directory with uniform naming structure;
    during lfp creation the directory and naming structure is used. 
    .'''
    
    # arcpy.conversion.FeatureClassToShapefile(smoothened_shp, save_dir)
    
    smoothened_shp = os.path.join(save_dir, 'Smoothened.shp')
    
    with arcpy.da.SearchCursor(smoothened_shp, ['OID@', "SHAPE@", 'Pegel_ID']) as cursor:
        for row in cursor:
            feature_id = row[0]
            geometry = row[1]
            pegel_id = row[2]
            output_filename = f"smoothened_{pegel_id}.shp"
            output_path = os.path.join(save_dir, output_filename)
    
            # Export the single feature to a shapefile
            try:
                arcpy.management.CopyFeatures(geometry, output_path)
                print(f"Exported feature {feature_id} to {output_path}")
            except arcpy.ExecuteError:
                print(arcpy.GetMessages(2))



# arcpy.env.addOutputsToMap = False #when working with multiple catchments, keeping == False increases speed. for debugging, keep = True.
# ids = [4833101]
# # get_smoothened_shp_ind() #doesn't need to be run after running once
# lfp_python(p_id= ids)



This calculates the longest flow path raster and converts it into polyline. 
Merit hydro and this has similar structure, but different relative file path structures used, hence rewritten.

In [ ]:
def lfp_python(p_id = None, 
               smoothened_shp = r'.\final.gdb\catchment_and_snappoints\Smoothened',
               dem_csv_path = r'.\python_outputs\lfp_outputs\1_Combined_Overview_Catchments_(Old_New).xlsx', 
               sheet_name = 'lfp_reconditioning',
               rn_1 = r'.\final.gdb\rn_ls_nrw_sa_merge',
               rn_2 = r'.\final.gdb\River_Net_LS_UTM32_new_ii',
               smoothened_cat_dir= r'.\python_outputs\lfp_outputs\1_4_smoothened_catchment',
               out_clipped_rn_dir = r'.\python_outputs\lfp_outputs\1_river_network_polyline',
               dem_reclass_dir = r'.\python_outputs\lfp_outputs\1_1_dem_reclass',
               dem_poly_dir = r'.\python_outputs\lfp_outputs\1_2_dem_polygon',
               out_rn_raster_dir = r'.\python_outputs\lfp_outputs\2_river_network_raster',
               out_agree_raster_dir = r'.\python_outputs\lfp_outputs\3_agree_dem',
               out_fill_dem_dir = r'.\python_outputs\lfp_outputs\4_fill_dem',
               out_fdr_dir = r'.\python_outputs\lfp_outputs\5_fdr',
               ds_fl_dir = r'.\python_outputs\lfp_outputs\6_ds',
               us_fl_dir = r'.\python_outputs\lfp_outputs\7_us',
               total_dir = r'.\python_outputs\lfp_outputs\8_total',
               clipped_dir = r'.\python_outputs\lfp_outputs\9_clipped total',
               extent_clipped_dir = r'.\python_outputs\lfp_outputs\9_clipped total_extent',
               lfp_raster_calc_dir = r'.\python_outputs\lfp_outputs\10_raster_calc_con',
               lfp_reclass_dir = r'.\python_outputs\lfp_outputs\11_reclass',
               lfp_polyline_dir = r'.\python_outputs\lfp_outputs\12_lfp',
               clipped_lfp_dir = r'.\python_outputs\lfp_outputs\13_clipped_lfp'
              ):
    '''
    This is similar to model: 6_LFP_raster. 
    This is used for Lower Saxony only (catchments delineated). 
    For Bavaria, the same method is used, but different notebook is used, as it starts from step 6 only, and the projection system used is also different.  
    Inputs:
        i. p_id: pegel_id of catchments to determine lfp for. if no p_id is given, this will delineate lfp for all catchments in the csv.
        ii. smoothened_shp: path of catchments, as explained in get_smoothened_shp_ind()
        iii. dem_csv_path: path of the csv. 
        iv. sheet_name: name of sheet in the csv; the structure of sheet is assumed as the first statement within for loop. 
        v. rn_1: path of the river network used for most catchments in lower saxony
        vi. rn_2: path of river network used for catchments [3692105, 4882139, 4882152, 4887121]
        vii. smoothened_cat_dir: smoothened catchments used for clipping rasters and polylines
        viii. rest are directories to be maintained presented chronologically with respect to the operation
    Outputs:
        Each step here creates an output. these outputs are named using {process}_{pegel_id}.{extension}
        final output is clipped_lfp. This is used to calculate time of concentration. 
    '''
    arcpy.env.overwriteOutput = True
    
    dem_csv = pd.read_excel(dem_csv_path, sheet_name = sheet_name, header = 0) 
    if p_id != None:
        dem_csv = dem_csv[dem_csv['Pegel_ID'].isin(p_id)]   #for when using list for multiple catchments
    # print(dem_csv.head())
    n_cats = len(dem_csv) 
    for cat_id in range(n_cats):
        #the structure of the sheet is assumed as follows. 
        walled_dem_path = dem_csv.iloc[cat_id, 0]
        pegel_id = int(dem_csv.iloc[cat_id, 3])
        buffer_cells = int(dem_csv.iloc[cat_id, 5])
        smooth_drop = int(dem_csv.iloc[cat_id, 6])
        sharp_drop = int(dem_csv.iloc[cat_id, 7])
        print(f'Pegel ID: {pegel_id}, buffer cells: {buffer_cells} , smooth drop: {smooth_drop}, sharp drop: {sharp_drop}')
        if pegel_id in [3692105, 4882139, 4882152, 4887121]:
            rn = rn_2
        else:
            rn = rn_1
        if buffer_cells == 0: 
            ##this fails dem reconditioning by default. so if certain catchments are not to be determined, its buffer cells can be kept 0 and skipped. 
            continue
        
        #1. clip dem by catchment polyline -> convert the dem into a polygon -> clip river network polyline -> rasterize polyline   
        #1.0 clip river polyline by walled dem 
        
        out_clipped_rn_name = f'rn_{pegel_id}.shp'
        out_clipped_rn_path = os.path.join(out_clipped_rn_dir,out_clipped_rn_name)
        
        
        
        #1.1 reclassifying dem into pollgon for clipping
        
        dem_reclass_file = f'reclass_dem_{pegel_id}.tif'
        dem_reclass_path = os.path.join(dem_reclass_dir, dem_reclass_file)
        temp_reclass = arcpy.sa.Reclassify(walled_dem_path, 'Value', RemapRange([[-10000, 10000, 1],
                                                                                 ["NODATA","NODATA", 'NODATA']]))
        temp_reclass.save(dem_reclass_path)
        print('1_Walled DEM raster reclassified to clipping polygon')
        
        #1.2 reclassified dem into polygon
        
        dem_poly_file = f'dem_poly_{pegel_id}.shp'
        dem_poly_path = os.path.join(dem_poly_dir, dem_poly_file)
        arcpy.conversion.RasterToPolygon(dem_reclass_path, dem_poly_path)
        print('2_Clipping Polygon Created')
        #clip river network with dem polygon
         
        
        arcpy.analysis.Clip(rn, dem_poly_path, out_clipped_rn_path)
        print('3_River network clipped by polygon')
        
        #1.4. Rasterize the polyline
         
        out_rn_raster_name = f'rn_{pegel_id}'
        out_clipped_rn_raster_path = os.path.join(out_rn_raster_dir, out_rn_raster_name)
        try:
            arcpy.conversion.PolylineToRaster(out_clipped_rn_path, 'FID', out_clipped_rn_raster_path, "MAXIMUM_LENGTH",'#' , 24.61, 'BUILD')
            print('4_River polyline converted into raster')
        except:
            print(f'4_River polyline to raster failed')
        
        
        #2. DEM Reconditionng
         
        out_agree_raster_name = f'agree_{pegel_id}.tif'
        out_agree_raster_path = os.path.join(out_agree_raster_dir, out_agree_raster_name)
        dem_recon_params = (walled_dem_path, out_clipped_rn_raster_path,
                           buffer_cells, smooth_drop, sharp_drop, out_agree_raster_path)
        p_demrecon = demreconditioning.DEMReconditioning()
        ret_demrecon  = p_demrecon.execute(dem_recon_params, None)
        print(f'5_DEM Reconditioning status: {ret_demrecon[0]}')

        #4. Fill DEM  ##for fill used the .sa instead of AH as in source code, sa is used if all parameters are none
         
        out_fill_raster_name = f'fill_{pegel_id}.tif'
        out_fill_raster_path = os.path.join(out_fill_dem_dir, out_fill_raster_name)
        
        ##errors here with fillAH, hence fill sa used
        fill_params = (out_agree_raster_path, out_fill_raster_path, None, None, None) #
        p_fill = fillsinks.FillSinks()
        p_fill.bCallFromPYT = False
        ret_fill  = p_fill.execute(fill_params,None)
        print(f'6_Fill status: {ret_fill[0]}')
        
        
        fill_dem = arcpy.sa.Fill(out_agree_raster_path, None)
        fill_dem.save(out_fill_raster_path)
        

        #5. Flow Direction 
        
        out_fdr_file = f'fdr_{pegel_id}.tif'
        fdr_path = os.path.join(out_fdr_dir, out_fdr_file)
        tmp_flow_dir = arcpy.sa.FlowDirection(out_fill_raster_path, "NORMAL") #when calling fdr_ah, pointer issue raise, couldnot resolve. #fdr_ah uses these lines under the hood
        tmp_flow_dir.save(fdr_path)
        print('7_FDR Status: OK')
        
        #6. DS and US flowlength
        
        
        out_ds_fl_file = f'ds_{pegel_id}.tif'
        
        
        out_us_fl_file = f'us_{pegel_id}.tif'
        
        out_ds_fl_dir = os.path.join(ds_fl_dir, out_ds_fl_file)
        out_us_fl_dir = os.path.join(us_fl_dir, out_us_fl_file)

        tmp_ds = arcpy.sa.FlowLength(fdr_path, "DOWNSTREAM", "") #ds calculate
        tmp_ds.save(out_ds_fl_dir) #us save

        tmp_us = arcpy.sa.FlowLength(fdr_path, "UPSTREAM", "") #us calculate
        tmp_us.save(out_us_fl_dir) #us save
        print('8_US DS done')
        
        #7. Flowlength = DS + US
        
        out_total_file = f'total_{pegel_id}.tif'
        out_fl_file_path = os.path.join(total_dir, out_total_file)
        tmp_fl = arcpy.sa.RasterCalculator([out_ds_fl_dir, out_us_fl_dir], ['x', 'y'], 'x+y')
        tmp_fl.save(out_fl_file_path)
        print('9_Flowlwength Calculated')
        
        #8.1. mask total flowlength raster 
            #this is used to get the maximum value of the flowlength raster within the catchment, but not used for conversion to polyline to avoid resampling
        clipping_geom_file_name = f'smoothened_{pegel_id}.shp'
        clipping_geom_file_path = os.path.join(smoothened_cat_dir, 
                                               clipping_geom_file_name)

        
        out_clip_dir = f'clipped_lfp_{pegel_id}.tif'
        out_lfp_clip_path = os.path.join(clipped_dir, out_clip_dir)
        
        temp_raster = arcpy.sa.ExtractByMask(out_fl_file_path, clipping_geom_file_path)
        temp_raster.save(out_lfp_clip_path)
        print('10_a_Flowlength raster clipped by catchment geometry')
        
        #8.2 Clipped flowlength by the spatial extent; creates a bounding box using rectangular extents around catchment
            #using this for polyline conversion increases runtime, but the raster will not be resampled 
            #also avoids the problem of missing line in some catchments
        
        
        extent_out_clip_file = f'clipped_lfp_{pegel_id}.tif'
        extent_out_lfp_clip_path = os.path.join(extent_clipped_dir, extent_out_clip_file)
        arcpy.Clip_management(in_raster=out_fl_file_path,
                              rectangle='#',  
                              out_raster=extent_out_lfp_clip_path,
                              in_template_dataset=clipping_geom_file_path,
                              nodata_value='1.79e+308',  
                              clipping_geometry='NONE', 
                              maintain_clipping_extent='NO_MAINTAIN_EXTENT')
        print('10_b_Flowlength raster clipped by catchment extent')
        
        #9. Using raster caclculator to convert all lfp pixels into 0 and 1 
    
            #get reclassifying threshold using raster from 8.1
        lfp_raster = arcpy.Raster(out_lfp_clip_path)   #use the raster clipped by polygon to calculate threshold
        arcpy.CalculateStatistics_management(lfp_raster)
        reclas_threshold = int(math.floor(lfp_raster.maximum)) - 1
        
        #use the threshold to reclassify all raster (from 8.2) into 0 and 1
        extent_raster = arcpy.Raster(extent_out_lfp_clip_path)
        
        lfp_raster_calc_file = f'recalc_lfp_{pegel_id}.tif'
        lfp_raster_calc_path = os.path.join(lfp_raster_calc_dir, lfp_raster_calc_file)
        temp_recalc = arcpy.sa.Con(extent_raster >= reclas_threshold, 1, 0)  #use the lfp raster clipped by extent
        temp_recalc.save(lfp_raster_calc_path)
        print('11_Flowlwength raster recalculated into 1 and 0')
        
        #10. Reclassify recalculated raster into 1 and nodata
        
        lfp_reclass_file = f'reclass_lfp_{pegel_id}.tif'
        lfp_reclass_path = os.path.join(lfp_reclass_dir, lfp_reclass_file)
        temp_reclass = arcpy.sa.Reclassify(lfp_raster_calc_path, 'Value', RemapValue([[0, "NODATA"],
                                                                                      [1, 1],
                                                                                      ["NODATA","NODATA"]]))
        temp_reclass.save(lfp_reclass_path)
        print('12_Recalculated Flowlength raster reclassified into 1 and NODATA')
        
        #11. save the raster as polyline 
        
        lfp_polyline_file = f'lfp_{pegel_id}.shp'
        lfp_polyline_path = os.path.join(lfp_polyline_dir, lfp_polyline_file)
        arcpy.conversion.RasterToPolyline(lfp_reclass_path, lfp_polyline_path, 'NODATA', '#', 'NO_SIMPLIFY')
        print('13_Raster converted into polyline')
        
        
        #12. Clip the polyline by catchment shapefile 
        
        clipped_lfp_file = f'clipped_lfp_{pegel_id}.shp'
        clipped_lfp_path = os.path.join(clipped_lfp_dir, clipped_lfp_file)
        arcpy.analysis.PairwiseClip(lfp_polyline_path, clipping_geom_file_path, clipped_lfp_path)
        print('14_LFP polyline clipped')

DFS for hydrologically plausible lfp. 

In [ ]:
def add_fields(path, fields):
    '''
    Adds fields in a shapefile/feature class. 
    '''
    for req_field in fields:
        if req_field not in [_.name for _ in arcpy.ListFields(path)]:
            arcpy.management.AddField(path, req_field, 'DOUBLE')

def find_all_paths_undirected(edges, start, ends, path=[]):
    """
    lfp from 1create_lfp is a polyline containing different lines. Each line is an edge, and each ends of a line are nodes.
     

    Input:
        i. start: node closest to outlet (merit_hydro drain points used as outlet)
        ii. ends: for a polyline, if a node is used only once and is not start, then it is taken as input node. this calculation is done in get_path_all_attributes()
        iii. Edges: [(starting_node1, ending_node1)]
    Output:
        In case of paths ending at multiple points, this provides nodes in each of the flowpath. 
    """
    path = path + [start]
    
    if start in ends:
        return [path]
    
    # Get all connected nodes (both directions)
    connected_nodes = []
    for edge in edges:
        if edge[0] == start:
            connected_nodes.append(edge[1])
        elif edge[1] == start:  # Check reverse direction
            connected_nodes.append(edge[0])
    
    if not connected_nodes:
        return []
    
    paths = []
    for neighbor in connected_nodes:
        if neighbor not in path:  # Avoid cycles
            new_paths = find_all_paths_undirected(edges, neighbor, ends, path)
            for new_path in new_paths:
                paths.append(new_path)
    return paths

def get_path_all_attributes(lfp_path, snappoint_path, final_lfp_path):
    '''This part gives all the attributes required to do a DFS for the lfp. 
    i. length of each edge
    ii. elevation of the node(s) of an edge if needed
    iii. starting point of the flowpath == node closest to the catchment outlet'''
    nodes_from = []
    nodes_to = []
    all_edges = []
    edge_lengths = dict()
    #edge_fids = dict()
    edge_2_outlet_dist = dict() 
    with arcpy.da.SearchCursor(snappoint_path, ['SHAPE@']) as point_cursor:
        for point_row in point_cursor:
            outlet_point = point_row[0]
            with arcpy.da.SearchCursor(lfp_path, ["SHAPE@",'from_node', 'to_node']) as cursor:
                for row in cursor:
                    edge = row[0]
                    
                    #creating edges for dfs
                    from_node, to_node = (row[1], row[2])
                    nodes_from.append(from_node)
                    nodes_to.append(to_node)
                    
                    #create edges
                    all_edges.append((from_node, to_node))
                    
                    #L
                    edge_lengths[(from_node, to_node)] = round(edge.length, 4) #outputs in m
                    
                    #finding the node closest to catchment outlet
                    outlet_distance = edge.distanceTo(outlet_point)
                    edge_2_outlet_dist[outlet_distance] = (from_node, to_node)
    
    #this part separates end nodes from intersection nodes
    nodes_all = nodes_from + nodes_to
    end_nodes = [_ for _ in nodes_all if nodes_all.count(_) == 1]
    int_nodes = list(set([_ for _ in nodes_all if _ not in end_nodes]))
    
    #getting starting node
    starting_edge = edge_2_outlet_dist[min(edge_2_outlet_dist.keys())] #returns tuple of (from,to) of edge closest to outlet
    start_node = list(set(starting_edge).intersection(set(end_nodes)))[0] #tuple and list to sets, and A intersection B.
    
    ends = [_ for _ in end_nodes if _ != start_node]
    
    # print(all_edges, start, ends)
    paths = find_all_paths_undirected(all_edges, start_node, ends, path=[])
    return paths, edge_lengths

def find_longest_path(paths, edge_lengths):
    '''From the paths, this function selects the path with the longest length.
    Inputs:
        i.paths: from find_all_paths_undirected()
        ii. edge_lengths: this is a dictionary containing length of edge e.g. for pathA: {(node1, node2): l1, (node2, node3): l2, (node4, node3): l3}
    Output:
        path with the maximum length. eg [node1, node2, node4]'''
    # print(paths, edge_lengths)
    length = dict()
    for path in paths:
        nodes = [(i,j) for i,j in zip(path[0:-1], path[1:])] + [(i,j) for i,j in zip(path[1:], path[0:-1])]
        path_length = sum([edge_lengths[node] for node in nodes if node in edge_lengths.keys()])
        length[path_length] = path
    max_length = max(length.keys())
    max_path = length[max_length] #dict cant hold more than one key with same value, so no need to deal with duplication
    return max_path

def update_lines_is_lfp(lfp_path, lfp_dfs):
    '''This part adds new field to lfp shapefile: is_lfp.
    If an edge in the polyline belongs to the longest flow path [from find_longest_path()], is_lfp =1, else = 0.
    Inputs:
        i. lfp_path = file path for the lfp.
        ii. lfp_dfs = lfp with node ids obtained after dfs
    Output:
        i. Updates the lfp shapefile such that if both nodes of a line in lfp polyline is in lfp_dfs, is_lfp = 1, else 0.   
    '''
    
    add_fields(lfp_path, ['is_lfp'])
    with arcpy.da.UpdateCursor(lfp_path, ['FID', 'is_lfp', 'from_node', 'to_node']) as cursor:
        for row in cursor:
            edge_fid = row[0]
            if row[-1] in lfp_dfs and row[-2] in lfp_dfs:
                row[1] = 1
            else:
                row[1] = 0
            cursor.updateRow(row)

def create_single_lfp(lfp_path, lfp_final_path):
    '''from the lfp created from lfp_merit_hydro() and updated by update_lines_is_lfp(), 
        the longest line is selected and dissolved into one line.'''
    where_clause = "is_lfp >= 1"
    temp_layer = r"temp_lfp"
    arcpy.management.MakeFeatureLayer(lfp_path, temp_layer, where_clause) #this selects the lines from polyline
    arcpy.management.Dissolve(temp_layer, lfp_final_path) #this is supposed to merge those polylines into one line

def get_elevation_end_points(final_lfp_path, cat_z_file_path, 
                             lfp_end_point_save_path, lfp_end_z_save_path):
    '''This converts line ends to points, and uses them to extract z values.
    The dem used for extracting elevation for lower saxony is the dem after reconditioning and filling. 
    The dem reconditioning parameters are provided in the csv.'''
    arcpy.management.FeatureVerticesToPoints(final_lfp_path, lfp_end_point_save_path, 'BOTH_ENDS')
    zs = []
    with arcpy.da.SearchCursor(lfp_end_point_save_path, ['SHAPE@XY']) as z_cursor:
        for z_row in z_cursor:
            point = z_row[0]
            print(type(point))
            z_point = f'{point[0]} {point[1]}'
            print(z_point)
            z_res = arcpy.management.GetCellValue(cat_z_file_path, z_point)
            z = float(z_res.getOutput(0).replace(',','.'))
            zs.append(z)
    z_div = max(zs)
    z_outlet = min(zs)
    return z_outlet, z_div

def update_single_lfp(final_lfp_path, 
                      pegel_id, 
                      cat_slope_file_path,
                      cat_z_file_path,
                     lfp_slope_save_path,
                     lfp_end_point_save_path, 
                     lfp_end_z_save_path):
    '''
    1. the dissolved lfp has no attributes. 
    this part adds pegel id, length, z_at end nodes, and average slope along the flow path.
    2. geodesic length of the lfp is calculated. 
    3. using elevations from 6b time of concentration is calculated.
    4. A redundant field for time of concentration from slope exists. Not removed to maintain indexing.
        This redundant field has values of -999.'''
    add_fields(final_lfp_path, ['length_m', 'z1_m', 'z2_m', 'S_perc', 'tc_dz', 'tc_S', 'Pegel_ID'])
    arcpy.management.CalculateGeometryAttributes(final_lfp_path, [['length_m','LENGTH']], 'METERS')
    
    #check without catchment average slope
    avrg_slope = -999 #round(get_lfp_slope(final_lfp_path, cat_slope_file_path, lfp_slope_save_path),4)
    #avrg_slope = None
    
    
    z_min, z_div = get_elevation_end_points(final_lfp_path, cat_z_file_path, lfp_end_point_save_path, lfp_end_z_save_path)
    with arcpy.da.UpdateCursor(final_lfp_path, ['length_m', 'z1_m', 'z2_m', 'S_perc', 'tc_dz', 'tc_S', 'Pegel_ID']) as cursor:
        for row in cursor:
            L = row[0]
            dZ = abs(z_div-z_min)
            tc_dz = 0.0195 * (L ** 0.77) * (abs(z_div-z_min)/L)**-0.385 
            tc_S = -999 #0.0195 * (L ** 0.77) * (avrg_slope/100)**-0.385
            row[1] = z_min
            row[2] = z_div
            row[3] = avrg_slope
            row[4] = tc_dz
            row[5] = tc_S
            row[6] = pegel_id
            cursor.updateRow(row)
    #this part might throw error, as variables are used after end of context window 
            #print('L, dZ, avrg_slope, tc_dz, tc_S', L, dZ, avrg_slope, tc_dz, tc_S)
            return L/1000, dZ, avrg_slope, tc_dz, tc_S

def update_tc(catchments_shp, 
              lfp_dir, 
              snappoint_dir, 
              final_lfp_dir,
              cat_slope_file_dir,
              cat_z_file_dir,
              lfp_slope_save_dir,
              lfp_end_point_save_dir,
              lfp_z_raster_save_dir,
              pegel_ids_2_calc):
    '''
    This is the main function. 
    Helper functions are described above in the order of application.
    Inputs:
        i. catchments_shp =  file path of the catchments of Bavaria. 
            Original catchments (without smoothening and simplification) are used here.
        ii. snappoint_dir = dir path of gauge points.
        iii. final_lfp_dir = dir path of lfp from lfp_merit_hydro()
        iv. cat_slope_file_dir = dir path of catchment slopes in %
        v. cat_z_file_dir = dir containing clipped HydroDEM for catchments
        vi. lfp_slope_save_dir = dir to save the slope raster masked by lfp
        vii. lfp_end_point_save_dir = dir to save ending nodes of a lfp as xy points
        viii. lfp_z_raster_save_dir = dir to save the ending xy points with extracted elevations 
        ix. pegel_ids_2_calc = selected catchments to calculate. this can be set to None, and the if statement erased to calculate tc for all catchments.
    '''
    add_fields(catchments_shp, ['length_lfp', 'dz_lfp', 'S_lfp', 'tc_s', 'tc_dZ'])
    with arcpy.da.UpdateCursor(catchments_shp, ['Pegel_ID', 'length_lfp', 'dz_lfp', 'S_lfp', 'tc_dZ','tc_s']) as cursor: #, 
        for row in cursor:
            pegel_id = int(row[0])
            
            lfp_file = f'clipped_lfp_{pegel_id}.shp'
            lfp_path = os.path.join(lfp_dir, lfp_file)
            snappoint_path = os.path.join(snappoint_dir, f'i_{pegel_id}.shp')
            if os.path.exists(lfp_path) and os.path.exists(snappoint_path) == False:
                print('Save snappoint for ', pegel_id)
                
            elif os.path.exists(lfp_path) and os.path.exists(snappoint_path) and pegel_id != 11432002:# and int(row[-1]) == 0:# and pegel_id == 4819102:
                if pegel_id in pegel_ids_2_calc:  ##this statement can be removed and below left indented to do all catchments 
                    print(pegel_id)
                    final_lfp_file = f'lfp_{pegel_id}.shp'
                    final_lfp_path = os.path.join(final_lfp_dir, final_lfp_file)
                    
                    catchment_slope_file = f'mask_slope_{pegel_id}.tif'
                    catchment_slope_file_path = os.path.join(cat_slope_file_dir, catchment_slope_file)
                    catchment_z_file = f'fill_{pegel_id}.tif'
                    catchment_z_file_path = os.path.join(cat_z_file_dir, catchment_z_file)    
                    lfp_slope_save_file = f'lfp_S_{pegel_id}.tif'
                    lfp_slope_save_path = os.path.join(lfp_slope_save_dir, lfp_slope_save_file)
                    lfp_end_point_save_file = f'lfp_end_points_{pegel_id}.shp'
                    lfp_end_point_save_path = os.path.join(lfp_end_point_save_dir, lfp_end_point_save_file)
                    lfp_z_raster_save_file = f'lfp_z_{pegel_id}.tif'
                    lfp_z_raster_save_path = os.path.join(lfp_z_raster_save_dir, lfp_z_raster_save_file)
                    
                    paths, edge_lengths = get_path_all_attributes(lfp_path, snappoint_path, final_lfp_path)
                    longest_path = find_longest_path(paths, edge_lengths)
                    update_lines_is_lfp(lfp_path, longest_path)
                    create_single_lfp(lfp_path, final_lfp_path)
                    row[1], row[2], row[3], row[4], row[5] = update_single_lfp(final_lfp_path, 
                                                                               pegel_id, 
                                                                               catchment_slope_file_path, 
                                                                               catchment_z_file_path, 
                                                                               lfp_slope_save_path,
                                                                               lfp_end_point_save_path,
                                                                               lfp_z_raster_save_path)
                    cursor.updateRow(row)




arcpy.env.addOutputsToMap = False #when working with multiple catchments, keeping == False increases speed. for debugging, keep = True.
arcpy.env.overwriteOutput = True
all_ids = [3644116]#, 4866111, 4869108, 4892106, 4894107, 4894119, 4894120, 4894136, 4898107]
for x in all_ids:
    ids = [x]
# get_smoothened_shp_ind() #doesn't need to be run after running once
    lfp_python(p_id = ids) # all directories kept as default 
    update_tc(catchments_shp = r'.\catchments_all.shp', 
              lfp_dir = r'.\13_clipped_lfp', 
              snappoint_dir = r'.\valid_name_snappoints',
             final_lfp_dir = r'.\14_a_corrected_merged_lfp',
             cat_slope_file_dir = r'.\7_mask_slope',
             cat_z_file_dir = r'.\4_filled',
              lfp_slope_save_dir = r'.\0merit_hydro\999 lfp final\5 lfp slope',
             lfp_end_point_save_dir = r'.\18_a_lfp_end_points',
              lfp_z_raster_save_dir = r'.\18_d_lfp_raster',
             pegel_ids_2_calc = ids)
    
    print()

Creates Downstream Raster for better visualization of flowpath across the catchment. Shows hydrological divisions better.

In [ ]:
def get_ds_outlet_cat(file):
    raster_vals = []
    with arcpy.da.SearchCursor(file, ['RASTERVALU']) as cursor:
        for row in cursor:
            raster_vals.append(float(row[0]))
    print(raster_vals)
    return min(raster_vals)

def get_ds_ls_re(cat_all, cat_dir, 
                 lfp_ends_dir, ds_dir, ls_ends_ds_dir,   
                 mask_save_dir, red_save_dir,
                 outlet_reduced_ds_dir, 
                 pegel_id_list = None):
    '''this is for lower saxony to get downstream length of the catchments
    for specific catchmets masking fdr causes no fdr in certain pixels.'''
    with arcpy.da.SearchCursor(cat_all, ['Pegel_ID']) as cursor:
        for row in cursor:
            pegel_id = int(row[0])
            if pegel_id in pegel_id_list: #pegel_id_list == None: # or #this can be removed and rest indented to left to do all catchments at once 
                #get the catchment polygon to create the mask by
                ds_raster = os.path.join(ds_dir, f"ds_{pegel_id}.tif") 
                cat_poly = os.path.join(cat_dir, f"smoothened_{pegel_id}.shp")
                
                #do the masking operation here
                temp_ras = arcpy.sa.ExtractByMask(ds_raster, cat_poly, 'INSIDE')
                mask_save = os.path.join(mask_save_dir, f"ds_{pegel_id}.tif")
                temp_ras.save(mask_save)

                #get the value of the points here 
                lfp_ends = os.path.join(lfp_ends_dir, f'lfp_end_points_{pegel_id}.shp')
                ls_ends_ds_path = os.path.join(ls_ends_ds_dir, f"lfp_ends_ds_{pegel_id}.shp")
                #first use the masked 
                arcpy.sa.ExtractValuesToPoints(lfp_ends, ds_raster, ls_ends_ds_path)
                ds_outlet = get_ds_outlet_cat(ls_ends_ds_path)
                print(ds_outlet)
                if int(ds_outlet) == -9999: #if outlet not within the catchment, use this
                    arcpy.sa.ExtractValuesToPoints(lfp_ends, ds_raster, ls_ends_ds_path)
                    raster_xRes = arcpy.management.GetRasterProperties(ds_raster, 'MINIMUM')
                    raster_x = round(float(raster_xRes.getOutput(0).replace(',', '.')),6)
                    print(raster_x)
                    ds_outlet = get_ds_outlet_cat(ls_ends_ds_path) - raster_x
                    
                #raster calculator subtract by minimum value 
                temp_ras1 = arcpy.sa.Minus(mask_save, ds_outlet)
                ds_save_path = os.path.join(outlet_reduced_ds_dir, f'ds_{pegel_id}.tif')
                temp_ras1.save(ds_save_path)

                #flowlength km
                temp_ras2 = temp_ras1/1000
                ds_km_save_path = os.path.join(red_save_dir, f'ds_{pegel_id}.tif')
                temp_ras2.save(ds_km_save_path)

all_ids = [3644116]#, 4866111, 4869108, 4892106, 4894107, 4894119, 4894120, 4894136, 4898107]
for x in all_ids:
    ids = [x]
    get_ds_ls_re(cat_all = r'.\catchments_all.shp',
                    lfp_ends_dir = r'.\18_a_lfp_end_points',
                    ds_dir = r'.\6_ds',
                    cat_dir = r'.\1_4_smoothened_catchment',
                    mask_save_dir = r'.\21_ds_ls_lfp_raster\1_mask_ds',
                    ls_ends_ds_dir = r'.\20_Downstream\ls_utm_32_ds_points',
                    red_save_dir = r'.\21_ds_ls_lfp_raster\6_div_dir',
                    outlet_reduced_ds_dir = r'.\21_ds_ls_lfp_raster\2_reduced_ds',
                    pegel_id_list=ids)